In [1]:
!pip install transformers torch pandas numpy scikit-learn tqdm prettytable matplotlib --quiet

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%%writefile ransomware_0_day_detection.py
# Your full script with saving added at the end

import pandas as pd
import torch
from torch import nn
from torch.optim import Adam
from torch.utils.data import Dataset as TorchDataset, DataLoader
from transformers import GPT2Tokenizer, GPT2Model
from tqdm import tqdm
from sklearn.metrics import classification_report
import argparse
import os

class Dataset(TorchDataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
        self.tokenizer.pad_token = self.tokenizer.eos_token

        self.texts = (
            self.df['apiFeatures'].fillna('') + " " +
            self.df['dropFeatures'].fillna('') + " " +
            self.df['regFeatures'].fillna('') + " " +
            self.df['filesFeatures'].fillna('') + " " +
            self.df['filesEXTFeatures'].fillna('') + " " +
            self.df['dirFeatures'].fillna('') + " " +
            self.df['strFeatures'].fillna('')
        ).str.strip().tolist()

        self.labels = (self.df['family'].astype(str) != '0').astype(int).tolist()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, truncation=True, max_length=1024, padding='max_length', return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

class Classifier(nn.Module):
    def __init__(self, hidden_size=768, num_classes=2, max_seq_len=1024,
                 gpt_model_name="zhouce/RDC-GPT", compression_ratio=128):
        super().__init__()
        self.gpt = GPT2Model.from_pretrained(gpt_model_name)
        self.linear = nn.Linear(hidden_size, num_classes)
        self.compression_ratio = compression_ratio

    def forward(self, input_ids, attention_mask):
        outputs = self.gpt(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state.mean(dim=1)
        logits = self.linear(pooled)
        return logits

def train(model, train_data, test_data, learning_rate=1e-5, epochs=20):
    train_dataset = Dataset(train_data)
    test_dataset = Dataset(test_data)

    train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=2)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=learning_rate)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            pred = outputs.argmax(dim=1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)

        train_acc = correct / total
        print(f"Epoch {epoch+1} | Train Loss: {total_loss/len(train_loader):.4f} | Train Acc: {train_acc:.4f}")

        model.eval()
        preds, trues = [], []
        total_acc_test = 0
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                outputs = model(input_ids, attention_mask)
                pred = outputs.argmax(dim=1)
                preds.extend(pred.cpu().tolist())
                trues.extend(labels.cpu().tolist())

                total_acc_test += (pred == labels).sum().item()

        test_acc = total_acc_test / len(test_data)
        print(f"Test Accuracy: {test_acc:.4f}")
        print(classification_report(trues, preds, target_names=['Goodware', 'Ransomware']))

        with open('result.txt', 'a') as f:
            f.write(f"Epoch {epoch+1} | Train Loss: {total_loss/len(train_loader):.4f} | Train Acc: {train_acc:.4f}\n")
            f.write(f"Test Accuracy: {test_acc:.4f}\n\n")

    # SAVE THE TRAINED MODEL HERE (to Google Drive)
    save_path = '/content/drive/MyDrive/srdc_finetuned_epoch20.pth'
    torch.save(model.state_dict(), save_path)
    print(f"Trained model saved to Google Drive: {save_path}")

def get_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--Data_Train_path', default='train.csv')
    parser.add_argument('--Data_Test_path', default='test.csv')
    return parser.parse_args()

if __name__ == "__main__":
    args = get_args()
    train_df = pd.read_csv(args.Data_Train_path)
    test_df = pd.read_csv(args.Data_Test_path)

    model = Classifier()
    train(model, train_df, test_df, epochs=20)

Overwriting ransomware_0_day_detection.py


In [5]:
!python ransomware_0_day_detection.py --Data_Train_path train.csv --Data_Test_path test.csv

config.json: 100% 944/944 [00:00<00:00, 5.55MB/s]
model.safetensors: 100% 498M/498M [00:05<00:00, 95.4MB/s]
Loading weights: 100% 148/148 [00:00<00:00, 1553.24it/s, Materializing param=wte.weight]
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 115kB/s]
vocab.json: 100% 1.04M/1.04M [00:00<00:00, 32.4MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 3.57MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 5.35MB/s]
Epoch 1/20: 100% 1219/1219 [05:33<00:00,  3.65it/s]
Epoch 1 | Train Loss: 0.4310 | Train Acc: 0.7949
Test Accuracy: 0.8918
              precision    recall  f1-score   support

    Goodware       0.96      0.87      0.91       200
  Ransomware       0.79      0.93      0.86       105

    accuracy                           0.89       305
   macro avg       0.88      0.90      0.88       305
weighted avg       0.90      0.89      0.89       305

Epoch 2/20: 100% 1219/1219 [05:44<00:00,  3.54it/s]
Epoch 2 | Train Loss: 0.2218 | Train Acc: 0.9188
Test Accuracy: 0.9443
        